# NumPy vectorization, broadcasting, einsum

Three canonical patterns: replace a Python loop with a vectorized op,
construct an outer product by broadcasting (without `np.outer`), and use
`np.einsum` for a batched inner product.

In [1]:
import time
import numpy as np

np.random.seed(0)

## 1. Loop vs vectorized: element-wise $\sin(x)^2 + \cos(x)^2$ on $10^6$ elements

Mathematically identically 1; the point is timing.

In [2]:
N = 1_000_000
x = np.random.uniform(0, 2 * np.pi, size=N)

t0 = time.perf_counter()
out_loop = [np.sin(v) ** 2 + np.cos(v) ** 2 for v in x]
t_loop = time.perf_counter() - t0

t0 = time.perf_counter()
out_vec = np.sin(x) ** 2 + np.cos(x) ** 2
t_vec = time.perf_counter() - t0

print(f"Python loop:       {t_loop:.4f} s")
print(f"Vectorized numpy:  {t_vec:.4f} s")
print(f"Speedup:           {t_loop / t_vec:.1f}x")
print(f"Max |loop - vec|:  {np.max(np.abs(np.array(out_loop) - out_vec)):.2e}")

Python loop:       6.1044 s
Vectorized numpy:  0.0757 s
Speedup:           80.6x
Max |loop - vec|:  2.22e-16


## 2. Broadcasting: outer product without `np.outer`

Rule: aligning shapes right-to-left, dimensions of size 1 stretch to match.
A column vector `(m, 1)` times a row vector `(n,)` broadcasts to `(m, n)`.

In [3]:
a = np.arange(1, 5)           # shape (4,)
b = np.arange(1, 4)           # shape (3,)

outer_bc = a[:, None] * b[None, :]   # (4,1) * (1,3) -> (4,3)
outer_ref = np.outer(a, b)

print("a =", a)
print("b =", b)
print("broadcast outer =")
print(outer_bc)
print("np.outer matches:", np.array_equal(outer_bc, outer_ref))

a = [1 2 3 4]
b = [1 2 3]
broadcast outer =
[[ 1  2  3]
 [ 2  4  6]
 [ 3  6  9]
 [ 4  8 12]]
np.outer matches: True


### What broadcasting is actually doing

`a[:, None]` has shape `(4, 1)`; `b[None, :]` has shape `(1, 3)`. Numpy
pretends each is `(4, 3)` by replicating along the 1-axes (without
allocating), then multiplies element-wise. Memory cost: $O(m + n)$,
not $O(mn)$, until the final result materialises.

## 3. `np.einsum`: batched inner product

Given a batch of $B = 100$ pairs of vectors in $\mathbb{R}^{64}$, compute
the $B$ inner products. Three equivalent spellings; einsum is the one
that scales without ceremony to tensor contractions.

In [4]:
B, D = 100, 64
U = np.random.randn(B, D)
V = np.random.randn(B, D)

dot_loop = np.array([U[i] @ V[i] for i in range(B)])
dot_sum  = (U * V).sum(axis=1)
dot_einsum = np.einsum("bd,bd->b", U, V)

print("loop vs einsum max diff: ", np.max(np.abs(dot_loop - dot_einsum)))
print("sum  vs einsum max diff: ", np.max(np.abs(dot_sum - dot_einsum)))
print("einsum result shape:     ", dot_einsum.shape)

loop vs einsum max diff:  5.329070518200751e-15
sum  vs einsum max diff:  7.105427357601002e-15
einsum result shape:      (100,)


The subscript `"bd,bd->b"` reads: "two inputs each indexed by `(b, d)`;
contract (sum over) `d`; keep `b` as the output axis." Same string pattern
generalises to e.g. attention: `"bhqd,bhkd->bhqk"`.

## Exercises

1. Replace the loop in exercise 1 with `np.vectorize` instead of a true
   vectorized op. Time it. Explain why it's not actually faster.
2. Compute pairwise squared Euclidean distances between two sets of points
   $X \in \mathbb{R}^{n \times d}$ and $Y \in \mathbb{R}^{m \times d}$
   using broadcasting only (no loops, no `scipy.spatial`).
3. Use `np.einsum` to compute a batched matrix–vector product
   `(B, m, n) @ (B, n) -> (B, m)`. Verify against a Python loop.